# Module 47 — Network Effect Analysis: Viral K-Factor

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, NumPy, Plotly  
> **Prerequisite**: Module 6 (Statistical Tests)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Referral Data](#3-synthetic-referral-data)
4. [K-Factor Calculation](#4-k-factor-calculation)
5. [Viral Growth Modeling](#5-viral-growth-modeling)
6. [Visualization](#6-visualization)
7. [Key Business Takeaway](#7-key-business-takeaway)

---
## §1 · Business Context

### Why measure viral growth?

Viral growth **reduces acquisition costs** through referrals:
- **K-factor > 1**: Each user brings more than 1 new user.
- **Exponential growth**: Viral loops accelerate user acquisition.
- **Cost efficiency**: Referrals are cheaper than paid acquisition.

**Applications**:
1. **Referral programs**: Measure program effectiveness.
2. **Product virality**: Track organic sharing.
3. **Growth forecasting**: Predict user growth.

**Key metrics**:
- **K-factor**: Invitations sent × Conversion rate.
- **Viral cycle time**: Time between invitation and signup.
- **Amplification factor**: Total users from one seed user.

In this notebook we will:
1. Generate synthetic referral chain data.
2. Calculate K-factor from referral metrics.
3. Model viral growth curves.
4. Identify tipping point for organic growth.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Referral Data

In [ ]:
# Generate synthetic referral data
N_USERS = 10000

# Referral parameters
avg_invitations = 5  # Average invitations per user
conversion_rate = 0.25  # 25% of invited users sign up

# Generate referral chains
referrals = []
for user_id in range(N_USERS):
    # Number of invitations sent
    n_invitations = np.random.poisson(avg_invitations)
    
    # Number of successful referrals
    n_conversions = np.random.binomial(n_invitations, conversion_rate)
    
    referrals.append({
        'user_id': user_id,
        'invitations_sent': n_invitations,
        'conversions': n_conversions,
        'conversion_rate': n_conversions / n_invitations if n_invitations > 0 else 0
    })

df = pd.DataFrame(referrals)

print(f"✓ Generated {N_USERS:,} users with referral data")
print(f"  Total invitations: {df['invitations_sent'].sum():,}")
print(f"  Total conversions: {df['conversions'].sum():,}")
print(f"  Overall conversion rate: {df['conversions'].sum() / df['invitations_sent'].sum()*100:.1f}%")

---
## §4 · K-Factor Calculation

In [ ]:
# Calculate K-factor
k_factor = avg_invitations * conversion_rate

print(f"✓ K-factor calculation")
print(f"  Average invitations per user: {avg_invitations}")
print(f"  Conversion rate: {conversion_rate*100:.1f}%")
print(f"  K-factor: {k_factor:.2f}")
print(f"\nInterpretation:")
if k_factor > 1:
    print(f"  ✅ K > 1: Viral growth! Each user brings {k_factor:.2f} new users.")
elif k_factor == 1:
    print(f"  ⚠️ K = 1: Linear growth. Each user brings exactly 1 new user.")
else:
    print(f"  ❌ K < 1: Sub-viral. Growth slows down over time.")

# Calculate amplification factor
amplification = 1 / (1 - k_factor) if k_factor < 1 else float('inf')
print(f"\nAmplification factor: {amplification:.2f}x")
print(f"  Each seed user eventually brings {amplification:.2f} total users.")

---
## §5 · Viral Growth Modeling

In [ ]:
# Model viral growth over time
N_CYCLES = 20
INITIAL_USERS = 100

# Growth model
growth = [INITIAL_USERS]
for cycle in range(1, N_CYCLES + 1):
    new_users = growth[-1] * k_factor
    growth.append(growth[-1] + new_users)

# Compare with paid acquisition
paid_growth = [INITIAL_USERS]
paid_users_per_cycle = 50  # Fixed paid acquisition
for cycle in range(1, N_CYCLES + 1):
    paid_growth.append(paid_growth[-1] + paid_users_per_cycle)

# Create comparison DataFrame
growth_df = pd.DataFrame({
    'cycle': range(N_CYCLES + 1),
    'viral': growth,
    'paid': paid_growth
})

print(f"✓ Growth model complete")
print(f"\nAfter {N_CYCLES} cycles:")
print(f"  Viral growth: {growth[-1]:,.0f} users")
print(f"  Paid growth: {paid_growth[-1]:,.0f} users")
print(f"  Viral advantage: {growth[-1] / paid_growth[-1]:.1f}x")

---
## §6 · Visualization

In [ ]:
# Visualize viral growth
fig = make_subplots(rows=1, cols=2, subplot_titles=['User Growth Comparison', 'K-Factor Sensitivity'])

# Growth comparison
fig.add_trace(go.Scatter(
    x=growth_df['cycle'],
    y=growth_df['viral'],
    mode='lines+markers',
    name='Viral Growth',
    line=dict(color='#EF553B', width=3)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=growth_df['cycle'],
    y=growth_df['paid'],
    mode='lines+markers',
    name='Paid Growth',
    line=dict(color='#636EFA', width=3)
), row=1, col=1)

# K-factor sensitivity
k_range = np.linspace(0.5, 1.5, 50)
final_users = [INITIAL_USERS / (1 - k) if k < 1 else 100000 for k in k_range]

fig.add_trace(go.Scatter(
    x=k_range,
    y=final_users,
    mode='lines',
    name='Total Users (after 20 cycles)',
    line=dict(color='#00CC96', width=3)
), row=1, col=2)

fig.add_vline(x=1.0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Viral growth (K>1) outpaces paid acquisition over time")
print("   - K-factor > 1 is the tipping point for exponential growth")
print("   - Small improvements in K-factor have large long-term impact")

---
## §7 · Key Business Takeaway

> ### 🎯 Action Items for Growth Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Referral programs, viral features, growth forecasting |
> | **Target** | K-factor > 1 for exponential growth |
> | **Levers** | Increase invitations OR conversion rate |
> | **Metrics** | K-factor, viral cycle time, amplification |
> | **Optimization** | A/B test referral incentives |
>
> ### 💡 Improving K-Factor
>
> | Lever | Strategy | Expected Impact |
> |-------|----------|------------------|
> | More invitations | Better referral UX | +20-30% invitations |
> | Higher conversion | Stronger incentives | +10-20% conversion |
> | Faster cycle | Real-time notifications | -50% cycle time |
>
> ### 🔑 Key Takeaways
>
> 1. **K-factor measures viral growth potential** quantitatively.
> 2. **K > 1 is the tipping point** for exponential growth.
> 3. **Viral growth outpaces paid acquisition** over time.
> 4. **Small K-factor improvements** have large long-term impact.
> 5. **Track K-factor regularly** to monitor referral program health.